# PGA Tour API rebuild

Builds `data/pga.db` from the PGA Tour's API and compares it with the hand-maintained
`data/golf.db`. **Reads golf.db, never writes it** — the weekly workflow in `pga-dk.ipynb`
is untouched.

Every table in `pga.db` is keyed by the Tour's own ids (`tournament_id` like `R2018013`,
`player_id` like `37455`); names live only in `players`. The file is safe to delete: the raw
API replies are cached under `data/api_cache/`, so a rebuild needs no network.

In [ ]:
import os, sys
from IPython import get_ipython

# Autoreload picks up edits to pga_api/ without a restart.
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("load_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

from pga_api import build, compare

## 1. Build `pga.db`

First run on a computer: about 5 minutes, fetching ~1,500 replies. After that it reads the cache
and takes seconds; only this season's schedule and stats are re-fetched.

What it prints per season: how many events the Tour lists and what kind each is —
**stroke** is the model's population (FedExCup events, plus the TOUR Championship);
**exhibition** is Hero, Olympics, Q-School and the team shootouts.

In [ ]:
frames = build.build(range(2015, 2027), stat_seasons=range(2014, 2027))

## 2. Compare with `golf.db`

Each line is a count of one kind of difference. The tables below it show the rows.

In [ ]:
r = compare.report()

### golf.db events filed under the wrong tournament

`golf_id` is what golf.db's `TOURN_ID` says; `tournament_id` is the event actually played that
week. Two golf.db events claiming one API event means one of them holds duplicated rows.

In [ ]:
ev = r["events"]
ev.loc[ev["id_wrong"] | ev["claimed_twice"],
       ["SEASON", "ENDING_DATE", "TOURNAMENT", "golf_rows", "golf_id", "tournament_id", "name", "field_size"]]

### Stroke-play events golf.db does not have

`source = leaderboard` means the Tour's past-results feed is empty for that event (renamed or
retired tournaments); those rows carry no FedExCup points or money.

In [ ]:
r["missing_events"]

### Round scores that differ

Where they differ, pga.db has been checked against hole-by-hole scorecards and matched them
every time. `format` is how golf.db stored the round (strokes before 2023, to-par after).

In [ ]:
r["rounds"].head(30)

### Season stats

In [ ]:
r["stats_summary"]